In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load the dataset

In [4]:
DATA_DIRECTORY = 'ncaa_box_scores_2024_25'
SCHEDULE_FILE = 'schedule.csv' 
SPLIT_DATE = '2025-02-28'
ROLLING_WINDOW_SIZE = 3

dataframes = []

for file_name in os.listdir(DATA_DIRECTORY):
    
    file_path = os.path.join(DATA_DIRECTORY, file_name)
    
    if os.path.isfile(file_path) and file_path.endswith('.csv'):
        
        df = pd.read_csv(file_path)
        if 'game_id' in df.columns and 'team' in df.columns and 'player' in df.columns:
             dataframes.append(df)

raw_dataframe = pd.concat(dataframes, ignore_index=True)
raw_dataframe

,player_id,player,MIN,FGM,FGA,3PTM,3PTA,FTM,FTA,OREB,...,team,opponent,home,starter,game_id,date,location,team_score,opponent_score,result
0,5177717.0,M. Gomma,26.0,3,4,0,0,1,2,1,...,Seattle U,Eastern Washington,False,True,401718465,2024-11-06,A,172,186,L
1,4610395.0,M. Moncrieffe,36.0,7,9,0,0,6,10,3,...,Seattle U,Eastern Washington,False,True,401718465,2024-11-06,A,172,186,L
2,5241342.0,B. Maldonado,14.0,2,9,2,5,1,1,0,...,Seattle U,Eastern Washington,False,True,401718465,2024-11-06,A,172,186,L
3,5107830.0,M. Arington,31.0,3,7,1,1,1,1,0,...,Seattle U,Eastern Washington,False,True,401718465,2024-11-06,A,172,186,L
4,4433606.0,J. Christofilis,31.0,4,11,4,6,5,6,0,...,Seattle U,Eastern Washington,False,True,401718465,2024-11-06,A,172,186,L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131998,5106735.0,J. Karasinski,15.0,2,6,1,3,0,0,0,...,Bellarmine,Florida Gulf Coast,False,False,401711649,2025-02-26,A,122,160,L
131999,5239765.0,M. Watkins,23.0,6,10,1,3,0,0,0,...,Bellarmine,Florida Gulf Coast,False,False,401711649,2025-02-26,A,122,160,L
132000,5175849.0,AJ Lux,23.0,3,5,2,4,0,0,1,...,Bellarmine,Florida Gulf Coast,False,False,401711649,2025-02-26,A,122,160,L
132001,4702669.0,D. McKinney,7.0,2,3,1,2,2,2,0,...,Bellarmine,Florida Gulf Coast,False,False,401711649,2025-02-26,A,122,160,L


# Cleaning + Preprocessing

In [5]:
team_dataframe = raw_dataframe[raw_dataframe['player'] == 'TEAM'].copy()

team_dataframe['date'] = pd.to_datetime(team_dataframe['date'])
team_dataframe = team_dataframe.sort_values(by=['date', 'game_id', 'team']) 

team_dataframe['result_numeric'] = (team_dataframe['result'] == 'W').astype(int)

cols_to_drop = ['player_id', 'player', 'MIN', 'starter', 'location', 'result']
team_dataframe = team_dataframe.drop(columns=cols_to_drop, errors='ignore')

team_dataframe

,FGM,FGA,3PTM,3PTA,FTM,FTA,OREB,DREB,REB,AST,...,PF,PTS,team,opponent,home,game_id,date,team_score,opponent_score,result_numeric
75471,20,55,9,31,3,8,9,22,31,13,...,16,52,American University,La Salle,False,401700387,2024-11-04,104,130,0
69562,23,55,5,16,14,18,12,28,40,7,...,14,65,La Salle,American University,True,401700387,2024-11-04,130,104,1
110416,24,67,6,22,9,11,14,25,39,10,...,26,63,Navy,Saint Joseph's,False,401700397,2024-11-04,126,140,0
51196,20,46,8,22,22,29,5,30,35,9,...,16,70,Saint Joseph's,Navy,True,401700397,2024-11-04,140,126,1
131653,26,56,4,13,9,13,12,18,30,8,...,23,65,Bellarmine,VCU,False,401700407,2024-11-04,130,168,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62917,29,72,8,31,32,37,17,27,44,12,...,27,98,Villanova,UCF,True,401758633,2025-04-05,196,208,0
82061,29,61,9,21,10,17,9,29,38,20,...,14,77,Nebraska,UCF,False,401758636,2025-04-06,154,132,1
27602,27,68,7,29,5,6,13,26,39,8,...,17,66,UCF,Nebraska,True,401758636,2025-04-06,132,154,0
101631,21,53,6,24,17,21,8,32,40,14,...,11,65,Florida,Houston,False,401746082,2025-04-07,130,126,1


In [7]:
opp_stats_temp = team_dataframe[[
    'game_id', 'team', 'FGM', 'FGA', '3PTM', '3PTA', 'FTM', 'FTA',
    'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS'
]].copy()
opp_stats_temp = opp_stats_temp.add_prefix('opp_')
opp_stats_temp.rename(columns={'opp_game_id': 'game_id', 'opp_team': 'opponent_team_actual'}, inplace=True)

team_dataframe = pd.merge(
    team_dataframe,
    opp_stats_temp,
    how='left',
    on='game_id',
    suffixes=('', '_y_drop')
)

team_dataframe

,FGM,FGA,3PTM,3PTA,FTM,FTA,OREB,DREB,REB,AST,...,opp_FTA_y_drop,opp_OREB_y_drop,opp_DREB_y_drop,opp_REB_y_drop,opp_AST_y_drop,opp_STL_y_drop,opp_BLK_y_drop,opp_TO_y_drop,opp_PF_y_drop,opp_PTS_y_drop
0,20,55,9,31,3,8,9,22,31,13,...,8,9,22,31,13,3,3,10,16,52
1,20,55,9,31,3,8,9,22,31,13,...,8,9,22,31,13,3,3,10,16,52
2,20,55,9,31,3,8,9,22,31,13,...,18,12,28,40,7,5,3,7,14,65
3,20,55,9,31,3,8,9,22,31,13,...,18,12,28,40,7,5,3,7,14,65
4,20,55,9,31,3,8,9,22,31,13,...,8,9,22,31,13,3,3,10,16,52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91910,24,69,6,25,9,14,15,24,39,5,...,14,15,24,39,5,5,6,9,19,63
91911,24,69,6,25,9,14,15,24,39,5,...,21,8,32,40,14,7,5,13,11,65
91912,24,69,6,25,9,14,15,24,39,5,...,21,8,32,40,14,7,5,13,11,65
91913,24,69,6,25,9,14,15,24,39,5,...,14,15,24,39,5,5,6,9,19,63


In [8]:
team_dataframe = team_dataframe[team_dataframe['opponent'] == team_dataframe['opponent_team_actual']].copy()

team_dataframe = team_dataframe.drop(columns=[col for col in team_dataframe.columns if '_y_drop' in col or col == 'opponent_team_actual'])


opp_allowed_cols_map = {col: col.replace('opp_', 'opp_allowed_') for col in opp_stats_temp.columns if col.startswith('opp_') and col != 'opp_game_id' and col != 'opp_team'}

team_dataframe.rename(columns=opp_allowed_cols_map, inplace=True)

team_dataframe

,FGM,FGA,3PTM,3PTA,FTM,FTA,OREB,DREB,REB,AST,...,opp_allowed_FTA,opp_allowed_OREB,opp_allowed_DREB,opp_allowed_REB,opp_allowed_AST,opp_allowed_STL,opp_allowed_BLK,opp_allowed_TO,opp_allowed_PF,opp_allowed_PTS
4,20,55,9,31,3,8,9,22,31,13,...,18,12,28,40,7,5,3,7,14,65
5,20,55,9,31,3,8,9,22,31,13,...,18,12,28,40,7,5,3,7,14,65
6,20,55,9,31,3,8,9,22,31,13,...,18,12,28,40,7,5,3,7,14,65
7,20,55,9,31,3,8,9,22,31,13,...,18,12,28,40,7,5,3,7,14,65
8,23,55,5,16,14,18,12,28,40,7,...,8,9,22,31,13,3,3,10,16,52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91906,21,53,6,24,17,21,8,32,40,14,...,14,15,24,39,5,5,6,9,19,63
91907,24,69,6,25,9,14,15,24,39,5,...,21,8,32,40,14,7,5,13,11,65
91908,24,69,6,25,9,14,15,24,39,5,...,21,8,32,40,14,7,5,13,11,65
91909,24,69,6,25,9,14,15,24,39,5,...,21,8,32,40,14,7,5,13,11,65


# Elo Rating Calculation

In [10]:
INITIAL_ELO = 1500
K_FACTOR = 20
HOME_ADVANTAGE_ELO = 100

def calculate_expected_outcome(rating1, rating2):
    return 1 / (1 + 10**((rating2 - rating1) / 400))

team_dataframe = team_dataframe.sort_values(by=['date', 'game_id', 'team']).reset_index(drop=True)

teams = pd.concat([team_dataframe['team'], team_dataframe['opponent']]).unique()
elo_ratings = {team: INITIAL_ELO for team in teams}
game_elos = {}

team_dataframe['team_elo_before'] = np.nan
team_dataframe['opp_elo_before'] = np.nan

unique_games = team_dataframe[['date', 'game_id']].drop_duplicates().sort_values(by=['date', 'game_id'])

for index, game_info in unique_games.iterrows():
    game_id = game_info['game_id']
    game_rows = team_dataframe[team_dataframe['game_id'] == game_id]

    if len(game_rows) != 2:
        continue

    row1 = game_rows.iloc[0]
    row2 = game_rows.iloc[1]

    team1 = row1['team']
    team2 = row1['opponent'] 

    elo1 = elo_ratings.get(team1, INITIAL_ELO)
    elo2 = elo_ratings.get(team2, INITIAL_ELO)

    team_dataframe.loc[row1.name, 'team_elo_before'] = elo1
    team_dataframe.loc[row1.name, 'opp_elo_before'] = elo2
    team_dataframe.loc[row2.name, 'team_elo_before'] = elo2
    team_dataframe.loc[row2.name, 'opp_elo_before'] = elo1 

    home_team = team1 if row1['home'] else team2
    away_team = team2 if row1['home'] else team1

    elo_home = elo_ratings.get(home_team, INITIAL_ELO) + HOME_ADVANTAGE_ELO
    elo_away = elo_ratings.get(away_team, INITIAL_ELO)

    expected_home_win = calculate_expected_outcome(elo_home, elo_away)

    actual_home_win = row1['result_numeric'] if row1['home'] else row2['result_numeric']

    update_factor_home = K_FACTOR * (actual_home_win - expected_home_win)
    elo_ratings[home_team] += update_factor_home
    elo_ratings[away_team] -= update_factor_home 

In [11]:
# Drop rows where Elo wasn't calculated (e.g., first game, etc.)
    
initial_rows = len(team_dataframe)
team_dataframe.dropna(subset=['team_elo_before', 'opp_elo_before'], inplace=True)
print(f"Dropped {initial_rows - len(team_dataframe)} rows with missing Elo ratings.")
print("Elo calculation complete.")
print(team_dataframe[['date', 'team', 'opponent', 'team_elo_before', 'opp_elo_before']].tail())

Dropped 45391 rows with missing Elo ratings.
Elo calculation complete.
Empty DataFrame
Columns: [date, team, opponent, team_elo_before, opp_elo_before]
Index: []


# Calculate Rolling Averages

In [13]:
stats_cols_base = ['FGM', 'FGA', '3PTM', '3PTA', 'FTM', 'FTA', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'team_score', 'opponent_score']
opp_allowed_cols = [col for col in team_dataframe.columns if col.startswith('opp_allowed_')]
all_stats_cols = stats_cols_base + opp_allowed_cols

rolling_feature_cols = []

for col in all_stats_cols:
    new_col_name = f'{col}_roll{ROLLING_WINDOW_SIZE}'
    team_dataframe[new_col_name] = team_dataframe.groupby('team')[col].transform(
        lambda x: x.shift(1).rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).mean()
    )
    rolling_feature_cols.append(new_col_name)

team_dataframe[f'FG%_roll{ROLLING_WINDOW_SIZE}'] = (team_dataframe[f'FGM_roll{ROLLING_WINDOW_SIZE}'] / team_dataframe[f'FGA_roll{ROLLING_WINDOW_SIZE}']).fillna(0)
team_dataframe[f'3PT%_roll{ROLLING_WINDOW_SIZE}'] = (team_dataframe[f'3PTM_roll{ROLLING_WINDOW_SIZE}'] / team_dataframe[f'3PTA_roll{ROLLING_WINDOW_SIZE}']).fillna(0)
team_dataframe[f'FT%_roll{ROLLING_WINDOW_SIZE}'] = (team_dataframe[f'FTM_roll{ROLLING_WINDOW_SIZE}'] / team_dataframe[f'FTA_roll{ROLLING_WINDOW_SIZE}']).fillna(0)
team_dataframe[f'opp_allowed_FG%_roll{ROLLING_WINDOW_SIZE}'] = (team_dataframe[f'opp_allowed_FGM_roll{ROLLING_WINDOW_SIZE}'] / team_dataframe[f'opp_allowed_FGA_roll{ROLLING_WINDOW_SIZE}']).fillna(0)
team_dataframe[f'opp_allowed_3PT%_roll{ROLLING_WINDOW_SIZE}'] = (team_dataframe[f'opp_allowed_3PTM_roll{ROLLING_WINDOW_SIZE}'] / team_dataframe[f'opp_allowed_3PTA_roll{ROLLING_WINDOW_SIZE}']).fillna(0)

rolling_feature_cols.extend([f'FG%_roll{ROLLING_WINDOW_SIZE}', f'3PT%_roll{ROLLING_WINDOW_SIZE}', f'FT%_roll{ROLLING_WINDOW_SIZE}',
                             f'opp_allowed_FG%_roll{ROLLING_WINDOW_SIZE}', f'opp_allowed_3PT%_roll{ROLLING_WINDOW_SIZE}'])

In [14]:
# Drop rows where rolling average couldn't be calculated (first N games)
initial_rows = len(team_dataframe)
team_dataframe.dropna(subset=[f'FGM_roll{ROLLING_WINDOW_SIZE}'], inplace=True)
print(f"Dropped {initial_rows - len(team_dataframe)} rows with missing rolling averages.")
print("Rolling averages calculated.")
print(team_dataframe[['date', 'team', 'opponent', f'PTS_roll{ROLLING_WINDOW_SIZE}', f'FG%_roll{ROLLING_WINDOW_SIZE}']].tail())

Dropped 0 rows with missing rolling averages.
Rolling averages calculated.
Empty DataFrame
Columns: [date, team, opponent, PTS_roll3, FG%_roll3]
Index: []


# Create Matchup DataFrame

In [22]:
cols_for_matchup = ['game_id', 'date', 'team', 'opponent', 'home', 'result_numeric',
                    'team_elo_before', 'opp_elo_before'] + rolling_feature_cols

matchup_data = team_dataframe[cols_for_matchup].copy()

matchup_merged = pd.merge(
    matchup_data,
    matchup_data,
    on='game_id',
    suffixes=('_team', '_opp')
)

matchups_final = matchup_merged[matchup_merged['team_team'] != matchup_merged['team_opp']].copy()
matchups_final = matchups_final[matchups_final['team_team'] == matchups_final['opponent_opp']].copy()

matchups_final['game_pair_id'] = matchups_final.apply(
    lambda row: f"{row['game_id']}_{'_'.join(sorted([str(row['team_team']), str(row['team_opp'])]))}", axis=1
)
matchups_final = matchups_final.drop_duplicates(subset='game_pair_id')
matchups_final.drop(columns=['game_pair_id'], inplace=True) 

matchups_final['label'] = matchups_final['result_numeric_team']

matchups_final

,date_team,team_team,opponent_team,home_team,result_numeric_team,team_elo_before_team,opp_elo_before_team,FGM_roll3_team,FGA_roll3_team,3PTM_roll3_team,...,opp_allowed_BLK_roll3_opp,opp_allowed_TO_roll3_opp,opp_allowed_PF_roll3_opp,opp_allowed_PTS_roll3_opp,FG%_roll3_opp,3PT%_roll3_opp,FT%_roll3_opp,opp_allowed_FG%_roll3_opp,opp_allowed_3PT%_roll3_opp,label


# Feature Engineering

In [23]:
feature_cols_final = []

In [24]:
matchups_final['elo_diff'] = matchups_final['team_elo_before_team'] - matchups_final['team_elo_before_opp']
feature_cols_final.append('elo_diff')

In [25]:
stats_to_diff = ['PTS', 'team_score', 'opponent_score', 'FG%', '3PT%', 'FT%', 'REB', 'AST', 'TO', 'STL', 'BLK', 'PF', 'opp_allowed_FG%', 'opp_allowed_PTS']
for stat in stats_to_diff:
    col_team = f'{stat}_roll{ROLLING_WINDOW_SIZE}_team'
    col_opp = f'{stat}_roll{ROLLING_WINDOW_SIZE}_opp'
    if col_team in matchups_final.columns and col_opp in matchups_final.columns:
         diff_col_name = f'{stat}_roll_diff'
         matchups_final[diff_col_name] = matchups_final[col_team] - matchups_final[col_opp]
         feature_cols_final.append(diff_col_name)

In [26]:
matchups_final['team_off_vs_opp_def_fg%'] = matchups_final[f'FG%_roll{ROLLING_WINDOW_SIZE}_team'] - matchups_final[f'opp_allowed_FG%_roll{ROLLING_WINDOW_SIZE}_opp']
feature_cols_final.append('team_off_vs_opp_def_fg%')

matchups_final['opp_off_vs_team_def_fg%'] = matchups_final[f'FG%_roll{ROLLING_WINDOW_SIZE}_opp'] - matchups_final[f'opp_allowed_FG%_roll{ROLLING_WINDOW_SIZE}_team']
feature_cols_final.append('opp_off_vs_team_def_fg%')

In [27]:
matchups_final['team_is_home'] = matchups_final['home_team'].astype(int)
feature_cols_final.append('team_is_home')

In [28]:
matchups_final[feature_cols_final] = matchups_final[feature_cols_final].fillna(0)

In [30]:
matchups_final

,date_team,team_team,opponent_team,home_team,result_numeric_team,team_elo_before_team,opp_elo_before_team,FGM_roll3_team,FGA_roll3_team,3PTM_roll3_team,...,AST_roll_diff,TO_roll_diff,STL_roll_diff,BLK_roll_diff,PF_roll_diff,opp_allowed_FG%_roll_diff,opp_allowed_PTS_roll_diff,team_off_vs_opp_def_fg%,opp_off_vs_team_def_fg%,team_is_home


In [31]:
split_timestamp = pd.Timestamp(SPLIT_DATE)

# Ensure date column is correct for splitting
date_col_to_use = 'date_team' if 'date_team' in matchups_final.columns else 'date'
if date_col_to_use not in matchups_final.columns:
    raise KeyError(f"Date column '{date_col_to_use}' not found for splitting.")

matchups_final[date_col_to_use] = pd.to_datetime(matchups_final[date_col_to_use])

train_df = matchups_final[matchups_final[date_col_to_use] < split_timestamp].copy()
test_df = matchups_final[matchups_final[date_col_to_use] >= split_timestamp].copy()

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

if train_df.empty or test_df.empty:
    raise ValueError("Train or test split resulted in empty DataFrame. Check SPLIT_DATE and data range.")

X_train = train_df[feature_cols_final]
y_train = train_df['label']
X_test = test_df[feature_cols_final]
y_test = test_df['label']

Train shape: (0, 108), Test shape: (0, 108)


ValueError: Train or test split resulted in empty DataFrame. Check SPLIT_DATE and data range.